# SOLiD-PINN v0.8.4 — Bayesian UQ on Colab
CPU runtime; select **Runtime → Run all**. Mount Drive when requested. For automatic GitHub return, add a fine-grained GitHub personal access token to Colab Secrets named `GITHUB_TOKEN`, enable notebook access, restrict it to the SOLiD-PINN repository with Contents read/write. No token is placed in a URL, source, output file, or Git config on disk. The notebook publishes only synthetic aggregate results to a new results branch; it never pushes to main. The final ZIP is also saved in Drive. This is analytic Bayesian UQ, not Bayesian PINN or experimental validation.

In [ ]:
import os, sys, subprocess, tempfile, pathlib, hashlib, json, platform, shutil, datetime, uuid, zipfile, base64
from google.colab import drive, userdata
from pathlib import Path
for name in ("OPENBLAS_NUM_THREADS","OMP_NUM_THREADS","MKL_NUM_THREADS","NUMEXPR_NUM_THREADS"): os.environ[name]="1"
drive.mount('/content/drive')
REPO="https://github.com/theerayutbo/SOLiD-PINN.git"
REV="adcdd4579103b8f80d9ab5b22586da373bfa4f54"
WORK=Path(tempfile.mkdtemp(prefix="solid-v084-"))
subprocess.run(["git","clone","--quiet","--no-checkout",REPO,str(WORK)],check=True)
def git(*args, **kw):
    return subprocess.run(["git",*args],cwd=WORK,check=True,**kw)
git("checkout","--quiet","--detach",REV)
assert git("rev-parse","HEAD",capture_output=True,text=True).stdout.strip()==REV
EXPECTED={"scripts/bayes_interface.py":"71de1e9009ecbb192c3c65055aa83d1659d0c48d","scripts/test_bayes_interface.py":"8c0cc248d0fe93451f7d0b24d476defdbd25130f"}
for path,expected in EXPECTED.items():
    assert git("hash-object",path,capture_output=True,text=True).stdout.strip()==expected
subprocess.run([sys.executable,"-m","pytest","-q","scripts/test_bayes_interface.py"],cwd=WORK,check=True)
sys.path.insert(0,str(WORK/"scripts"))
import bayes_interface as b
import numpy as np, scipy
from threadpoolctl import threadpool_limits
threadpool_limits(1)
def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
CONFIG={"source_revision":REV,"source_sha256":sha(WORK/"scripts/bayes_interface.py"),"protocol":"v084-original-108-checkpoint-v1","n":151,"draws":16000,"repeats":30,"environment":{"python":platform.python_version(),"numpy":np.__version__,"scipy":scipy.__version__}}
RUN_ID=hashlib.sha256(json.dumps(CONFIG,sort_keys=True).encode()).hexdigest()[:16]
OUT=Path("/content/drive/MyDrive/SOLiD-PINN/v084_colab")/RUN_ID
CHECK=OUT/"checkpoints";CHECK.mkdir(parents=True,exist_ok=True)
def atomic_json(path,obj):
    tmp=path.with_name(path.name+".tmp")
    tmp.write_text(json.dumps(obj,indent=2,allow_nan=False)+"\n")
    tmp.replace(path)
atomic_json(OUT/"config.json",CONFIG)
print("Verified source and tests. Run ID:",RUN_ID)

## Execute original 108 posterior calculations with per-case checkpointing
Original representative and refinement seeds and coverage seeds are preserved. Each completed posterior is saved as an aggregate JSON record. If disconnected, rerun all using the same runtime package versions; the run ID isolates changes in environment/configuration. Do not edit checkpoint files.

In [ ]:
N,D,REP=CONFIG["n"],CONFIG["draws"],CONFIG["repeats"]
rows=[];checks=[];coverage=[]
def valid_record(v):
    q=np.asarray(v["quantiles"],float); z=np.asarray(v["pred_quantiles"],float)
    return (q.shape==(3,4) and z.shape==(3,2) and np.isfinite(q).all() and np.isfinite(z).all() and np.all(np.diff(q,axis=0)>=0) and np.all(np.diff(z,axis=0)>=0) and 0<v["acceptance"]<=1 and len(v["covered"])==4 and len(v["pred_covered"])==2)
def one(name,sigma,prior,n,draws,noise_seed,post_seed):
    spec={"design":name,"noise":sigma,"prior":prior,"n":n,"draws":draws,"noise_seed":noise_seed,"post_seed":post_seed}
    key=hashlib.sha256(json.dumps(spec,sort_keys=True).encode()).hexdigest()[:20]
    path=CHECK/(key+".json")
    if path.exists():
        saved=json.loads(path.read_text())
        if saved["run_id"]!=RUN_ID or saved["spec"]!=spec or not valid_record(saved["result"]): raise RuntimeError("Invalid checkpoint "+key)
        return saved["result"]
    t,j=b.design(name)
    y=b.voltage(t,j,b.P)+np.random.default_rng(noise_seed).normal(0,sigma,len(t))
    p,m=b.posterior(t,j,y,sigma,prior,n,draws,post_seed)
    result=b.summarize(p,m)
    if not valid_record(result):raise RuntimeError("Invalid posterior "+key)
    atomic_json(path,{"run_id":RUN_ID,"spec":spec,"result":result})
    return result
for name in ("single","matched","multi"):
    for sigma in (1.,5.):
        for prior in ("uniform","log_j0"):
            m=one(name,sigma,prior,N,D,11,123)
            rows.append(dict(design=name,noise=sigma,prior=prior,**m))
            if prior=="uniform":
                m2=one(name,sigma,prior,2*N-1,2*D,11,456)
                coarse=np.array(m["quantiles"]); fine=np.array(m2["quantiles"])
                checks.append(dict(design=name,noise=sigma,coarse=coarse.tolist(),fine=fine.tolist(),delta_over_width=(np.abs(coarse-fine)/np.maximum(1e-12,np.ptp(fine[[0,2]],axis=0))).tolist()))
    for seed in range(100,100+REP):
        m=one(name,5.,"uniform",N,D,seed,seed+10000)
        coverage.append(dict(design=name,seed=seed,**m))
    print(name,"complete",len(rows)+len(checks)+len(coverage),"/108",flush=True)
assert (len(rows),len(checks),len(coverage))==(12,6,90)
result=dict(truth=b.P.tolist(),model="analytic v0.8.3 symmetric-cell ODE",n=N,draws=D,repeats=REP,coverage_noise_mv=5,representative_seed=11,parameters=["R0","Rinf","tau","j0"],environment=CONFIG["environment"],representative=rows,grid_checks=checks,coverage=coverage)
atomic_json(OUT/"results.json",result)
summary={"run_id":RUN_ID,"posterior_calculations":108,"tests":"3 passed (pytest exit 0)","max_grid_quantile_delta_over_fine_width":max(float(np.max(c["delta_over_width"])) for c in checks),"coverage_counts":{}}
for name in ("single","matched","multi"):
    g=[x for x in coverage if x["design"]==name]
    summary["coverage_counts"][name]={"n":len(g),"parameter":np.sum([x["covered"] for x in g],axis=0).tolist(),"latent_voltage":np.sum([x["pred_covered"] for x in g],axis=0).tolist()}
atomic_json(OUT/"summary.json",summary)
manifest=dict(CONFIG,run_id=RUN_ID,scope="synthetic aggregate results only; not experimental validation",status="complete",files={p:sha(OUT/p) for p in ("results.json","summary.json")},completed_utc=datetime.datetime.now(datetime.timezone.utc).isoformat())
atomic_json(OUT/"manifest.json",manifest)
print(json.dumps(summary,indent=2))

## Return results automatically to GitHub
Only three allowlisted aggregate JSON files are committed. No posterior draws, private files, checkpoint directory, environment variables, or token. A fresh results branch prevents overwriting other runs. If the secret is absent or GitHub denies access, the ZIP remains available on Drive. Enable notebook access to the token before running this cell.

In [ ]:
FILES=("results.json","summary.json","manifest.json")
assert json.loads((OUT/"manifest.json").read_text())["status"]=="complete"
for name,digest in manifest["files"].items(): assert sha(OUT/name)==digest
ZIP=OUT/"SOLiD-PINN_v084_colab_results.zip"
with zipfile.ZipFile(ZIP.with_suffix(".tmp"),"w",zipfile.ZIP_DEFLATED) as z:
    for name in FILES:z.write(OUT/name,name)
ZIP.with_suffix(".tmp").replace(ZIP)
print("Saved ZIP:",ZIP)
def publish():
    try: token=userdata.get("GITHUB_TOKEN")
    except Exception:
        print("GitHub secret unavailable. ZIP saved; enable GITHUB_TOKEN notebook access and rerun this cell.");return
    if not token:
        print("GitHub secret empty. ZIP saved.");return
    branch="results/v084-"+RUN_ID+"-"+uuid.uuid4().hex[:8]
    # Start from the verified source, regardless of earlier attempts.
    git("checkout","--quiet","--detach",REV)
    dest=WORK/"docs"/"results"/"colab"/RUN_ID
    dest.mkdir(parents=True,exist_ok=True)
    paths=[]
    for name in FILES:
        shutil.copyfile(OUT/name,dest/name);paths.append(str((dest/name).relative_to(WORK)))
    git("add","--",*paths)
    staged=git("diff","--cached","--name-only",capture_output=True,text=True).stdout.splitlines()
    if set(staged)!=set(paths):raise RuntimeError("Unexpected staged files; not publishing")
    git("-c","user.name=SOLiD-PINN Colab","-c","user.email=solid-pinn-colab@users.noreply.github.com","commit","--quiet","-m","Archive synthetic Bayesian UQ run "+RUN_ID)
    # Ephemeral process environment for authentication only; no token written to disk.
    auth=base64.b64encode(("x-access-token:"+token).encode()).decode()
    env=os.environ.copy()
    env.update({"GIT_CONFIG_COUNT":"1","GIT_CONFIG_KEY_0":"http.https://github.com/.extraheader","GIT_CONFIG_VALUE_0":"Authorization: Basic "+auth,"GIT_TERMINAL_PROMPT":"0"})
    p=subprocess.run(["git","push",REPO,"HEAD:refs/heads/"+branch],cwd=WORK,env=env,capture_output=True,text=True)
    if p.returncode:
        print("GitHub push failed. Check token repository access / Contents read-write. ZIP retained.");return
    commit=git("rev-parse","HEAD",capture_output=True,text=True).stdout.strip()
    receipt={"branch":branch,"commit":commit,"run_id":RUN_ID,"url":"https://github.com/theerayutbo/SOLiD-PINN/tree/"+branch+"/docs/results/colab/"+RUN_ID}
    atomic_json(OUT/"github_receipt.json",receipt)
    print("GitHub results:",receipt["url"])
publish()

If publishing succeeds, share the printed GitHub URL in ChatGPT (or ask ChatGPT to inspect results branches). If it fails, upload the saved ZIP. No automatic background notification is implied.